# Fine-tune Qwen2.5 Instruct with QLoRA

Notebook này dùng để chạy trên Google Colab.

Đầu vào mặc định là file:

```text
finetune_qa_80_20.jsonl
```

Model mặc định:

```text
Qwen/Qwen2.5-3B-Instruct
```

Nếu Colab bị thiếu VRAM, đổi sang:

```text
Qwen/Qwen2.5-1.5B-Instruct
```

## 1. Install Dependencies

In [ ]:
!pip install -q -U transformers accelerate datasets peft trl bitsandbytes sentencepiece

## 2. Upload Dataset

Upload file `processed/finetune_qa_80_20.jsonl` từ máy của bạn lên Colab.

In [ ]:
from google.colab import files

uploaded = files.upload()
DATA_PATH = next(iter(uploaded.keys()))
DATA_PATH

Nếu bạn dùng Google Drive thay vì upload thủ công, có thể bỏ qua cell upload và dùng đoạn dưới:

```python
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/finetune_qa_80_20.jsonl'
```

## 3. Load and Check Dataset

In [ ]:
import json
from pathlib import Path

records = []
with open(DATA_PATH, "r", encoding="utf-8") as file:
    for line in file:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print("Records:", len(records))
records[0]

In [ ]:
def validate_record(record):
    assert "messages" in record
    messages = record["messages"]
    assert isinstance(messages, list)
    assert len(messages) >= 3
    roles = [message.get("role") for message in messages]
    assert roles[0] == "system"
    assert "user" in roles
    assert roles[-1] == "assistant"
    for message in messages:
        assert message.get("content", "").strip()


for record in records[:100]:
    validate_record(record)

print("Validation passed for first 100 records")

## 4. Convert to Hugging Face Dataset

In [ ]:
from datasets import Dataset

raw_dataset = Dataset.from_list(records)
split_dataset = raw_dataset.train_test_split(test_size=0.05, seed=42)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(train_dataset)
print(eval_dataset)

## 5. Load Qwen2.5 Model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
# MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # Use this if Colab runs out of VRAM.

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False

## 6. Format Chat Data

In [ ]:
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


train_dataset = train_dataset.map(format_chat, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(format_chat, remove_columns=eval_dataset.column_names)

print(train_dataset[0]["text"][:1200])

## 7. Configure LoRA

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

## 8. Train

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR = "/content/qwen2_5_shopping_chatbot_lora"
MAX_SEQ_LENGTH = 2048

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    peft_config=lora_config,
    args=training_args,
)

trainer.train()

Nếu `SFTTrainer` báo lỗi tham số `dataset_text_field` hoặc `max_seq_length`, bạn đang dùng phiên bản `trl` mới hơn. Dùng cell thay thế dưới đây.

In [ ]:
# Alternative for newer TRL versions. Run only if the previous trainer cell fails.
# from trl import SFTConfig, SFTTrainer
#
# sft_config = SFTConfig(
#     output_dir=OUTPUT_DIR,
#     num_train_epochs=2,
#     per_device_train_batch_size=1,
#     per_device_eval_batch_size=1,
#     gradient_accumulation_steps=8,
#     learning_rate=2e-4,
#     warmup_ratio=0.03,
#     lr_scheduler_type="cosine",
#     logging_steps=10,
#     eval_strategy="steps",
#     eval_steps=100,
#     save_steps=100,
#     save_total_limit=2,
#     fp16=True,
#     optim="paged_adamw_8bit",
#     report_to="none",
#     max_seq_length=MAX_SEQ_LENGTH,
#     dataset_text_field="text",
# )
#
# trainer = SFTTrainer(
#     model=model,
#     processing_class=tokenizer,
#     train_dataset=train_dataset,
#     eval_dataset=eval_dataset,
#     peft_config=lora_config,
#     args=sft_config,
# )
# trainer.train()

## 9. Save Adapter

In [ ]:
ADAPTER_DIR = "/content/qwen2_5_shopping_chatbot_adapter"

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Saved adapter to:", ADAPTER_DIR)

## 10. Quick Inference Test

In [ ]:
from peft import PeftModel

test_messages = [
    {
        "role": "system",
        "content": "Bạn là chatbot tư vấn mua sắm. Trả lời ngắn gọn, chính xác, không bịa thông tin.",
    },
    {
        "role": "user",
        "content": "Tư vấn cho tôi một điện thoại RAM 8GB.",
    },
]

prompt = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output_ids = trainer.model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

## 11. Download Adapter

In [ ]:
!zip -r /content/qwen2_5_shopping_chatbot_adapter.zip /content/qwen2_5_shopping_chatbot_adapter

from google.colab import files
files.download('/content/qwen2_5_shopping_chatbot_adapter.zip')